In [1]:
import pandas as pd 
import numpy as np 

## 1. Clientes por estado y ciudad

Representa una clasificación del número de clientes por estado. Crea una tabla en la que se muestren:

- Estado
- Ciudad
- Número de clientes por ciudad

Tanto la tabla como los gráficos deberán ser dinámicos respecto a la fecha para permitir el análisis temporal de la evolución de clientes.

In [2]:

dfCustomersBase = pd.read_csv('./streamlit_resources/customers_dataset.csv')
dfOrdersBase = pd.read_csv('./streamlit_resources/orders_dataset.csv')
#dfCustomersBase[['customer_state','customer_city','customer_id']].groupby(['customer_city','customer_state']).count()

dfMergeOrdersCustomers = pd.merge(dfCustomersBase,dfOrdersBase, on='customer_id')
dfMergeOrdersCustomers['order_purchase_timestamp'] = pd.to_datetime(dfMergeOrdersCustomers.order_purchase_timestamp, yearfirst=True)
dfFinalEj1 = dfMergeOrdersCustomers[['customer_state','customer_id','customer_city','customer_unique_id']].where((dfMergeOrdersCustomers['order_purchase_timestamp'] > pd.to_datetime('2013-01-01 00:00:00', yearfirst=True)) &( dfMergeOrdersCustomers['order_purchase_timestamp'] < pd.to_datetime('2020-01-01 00:00:00', yearfirst=True))).groupby(['customer_city','customer_state']).nunique().sort_values(['customer_unique_id'], ascending=False).reset_index()
dfFinalEj1[['customer_state','customer_city','customer_unique_id', 'customer_id']]

,customer_state,customer_city,customer_unique_id,customer_id
0,SP,sao paulo,14984,15540
1,RJ,rio de janeiro,6620,6882
2,MG,belo horizonte,2672,2773
3,DF,brasilia,2069,2131
4,PR,curitiba,1465,1521
...,...,...,...,...
4305,RS,gramado dos loureiros,1,1
4306,AP,vitoria do jari,1,1
4307,PE,xexeu,1,1
4308,PR,adrianopolis,1,1


## 2. Pedidos por ciudad

A la tabla anterior añade las siguientes columnas:

- Número de pedidos
- Porcentaje que representan respecto al total de pedidos

Además, representa el ratio de pedidos por cliente, utilizando el tipo de gráfico que consideres más adecuado.

Tras este análisis, responde a las siguientes cuestiones:

- ¿Qué información o patrones se pueden identificar a partir de estos datos?
- ¿Qué acciones, como analista de datos, crees que debería tomar la empresa para mejorar sus ventas?

In [3]:

dfOrdersCustomersCity = dfFinalEj1.copy()
dfOrdersCustomersCity = dfOrdersCustomersCity.rename(columns={'customer_id':'orders_count','customer_unique_id':'customer_count'})
dfOrdersCustomersCity['orders_percent'] = 0.0
dfOrdersCustomersCity['orders_percent'] = round(dfOrdersCustomersCity['orders_count'] / dfOrdersCustomersCity['orders_count'].sum() * 100, 3)
dfOrdersCustomersCity['proporcion_pedidos_cliente'] = round(dfOrdersCustomersCity['orders_count'] / dfOrdersCustomersCity['customer_count'],2)
dfOrdersCustomersCity



,customer_city,customer_state,orders_count,customer_count,orders_percent,proporcion_pedidos_cliente
0,sao paulo,SP,15540,14984,15.627,1.04
1,rio de janeiro,RJ,6882,6620,6.921,1.04
2,belo horizonte,MG,2773,2672,2.789,1.04
3,brasilia,DF,2131,2069,2.143,1.03
4,curitiba,PR,1521,1465,1.530,1.04
...,...,...,...,...,...,...
4305,gramado dos loureiros,RS,1,1,0.001,1.00
4306,vitoria do jari,AP,1,1,0.001,1.00
4307,xexeu,PE,1,1,0.001,1.00
4308,adrianopolis,PR,1,1,0.001,1.00


## 3. Análisis de retrasos en pedidos

Calcula y representa:

- Número de pedidos que llegan tarde por ciudad
- Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad
- Tiempo medio de retraso en días

Además, al representar esta información, el dashboard deberá incluir un autodiagnóstico que indique la razón más probable del problema.

In [4]:

dfRetrasos = dfMergeOrdersCustomers.copy()

dfRetrasos['pedidos_tarde'] = (
    pd.to_datetime(dfRetrasos['order_delivered_customer_date']) > 
    pd.to_datetime(dfRetrasos['order_estimated_delivery_date'])
)

dfTotalRetrasosCiudad = dfRetrasos.groupby('customer_city')['pedidos_tarde'].sum().reset_index().sort_values('pedidos_tarde', ascending=False)

dfTotalRetrasosCiudadCopia = dfTotalRetrasosCiudad.copy()
dfOrdersCustomersCity = dfOrdersCustomersCity.copy()

dfTotalRetrasosCiudadCopia = pd.merge(dfTotalRetrasosCiudadCopia, dfOrdersCustomersCity[['customer_city', 'orders_count']], on='customer_city', how='left')

dfTotalRetrasosCiudadCopia['porcentaje_retrasados'] = round((dfTotalRetrasosCiudadCopia['pedidos_tarde'] / dfTotalRetrasosCiudadCopia['orders_count']) * 100, 2)

dfTotalRetrasosCiudadCopia.sort_values(by=['pedidos_tarde','porcentaje_retrasados'], ascending=[False, False])

,customer_city,pedidos_tarde,orders_count,porcentaje_retrasados
0,sao paulo,942,15540,6.06
1,rio de janeiro,780,6882,11.33
2,salvador,208,1245,16.71
3,belo horizonte,166,2773,5.99
4,porto alegre,158,1379,11.46
...,...,...,...,...
4305,acailandia,0,7,0.00
4306,acaiaca,0,2,0.00
4307,abreu e lima,0,11,0.00
4308,abre campo,0,6,0.00


In [5]:
dfTodosPedidosTiempo = dfMergeOrdersCustomers.copy()
dfTodosPedidosTiempo['tiempo_retraso'] = ((
    pd.to_datetime(dfTodosPedidosTiempo['order_delivered_customer_date']) - 
    pd.to_datetime(dfTodosPedidosTiempo['order_estimated_delivery_date'])
).dt.total_seconds() / (24 * 3600)).round()

dfTodosPedidosTiempoTarde = dfTodosPedidosTiempo[dfTodosPedidosTiempo['tiempo_retraso'] > 0]
dfTodosPedidosTiempoTardeCity = dfTodosPedidosTiempoTarde.groupby('customer_city')['tiempo_retraso'].mean().reset_index().sort_values('tiempo_retraso', ascending=False).rename(columns={'tiempo_retraso': 'tiempo_retraso_medio'})

dfTodosPedidosTiempoTardeCity

,customer_city,tiempo_retraso_medio
723,montanha,182.0
853,perdizes,163.0
656,macapa,145.0
769,novo brasil,127.0
929,quintana,122.0
...,...,...
63,ararangua,1.0
1212,uaua,1.0
1260,viradouro,1.0
23,alvares florence,1.0


## 4. Reviews y satisfacción del cliente

Calcula y representa:

- Número de reviews por estado
- Score medio de las reviews en cada estado

Para este cálculo, se deberán excluir los pedidos con retraso, ya que se entiende que la valoración negativa podría deberse principalmente al retraso en la entrega del producto.

Esto serán las métricas que tendrá que tener en el ejercicio calculadas y representadas como mínimo, puedes añadir todas las que veas interesantes!

In [6]:

dfReviews = pd.read_csv('./streamlit_resources/order_reviews_dataset.csv')
dfMergeReviewsCustomers = pd.merge(dfReviews, dfMergeOrdersCustomers, on='order_id')

dfMergeReviewsCustomers = pd.merge(dfMergeReviewsCustomers,dfRetrasos[['order_id','pedidos_tarde']], on='order_id')

df_estado_reviews = dfMergeReviewsCustomers[dfMergeReviewsCustomers['pedidos_tarde'] == False].groupby('customer_state').agg({'review_score': 'mean','review_id': 'count'}).rename(columns={'review_score':'score_average','review_id':'review_count'})

df_estado_reviews.sort_values(by='score_average', ascending=False)


,score_average,review_count
customer_state,,
RN,4.344907,432
MS,4.308411,642
RS,4.254852,5101
SP,4.249650,39335
PR,4.244421,4795
AM,4.241135,141
TO,4.225410,244
SC,4.223339,3282
MG,4.219565,10999
